# Building a RAG Application — Logic Notebook

Companion notebook for **IM Session 32 - Building a RAG Application**.

This notebook develops and tests the retrieval + generation logic used by the Streamlit app (`Building a RAG Application - app.py`) in this same folder, before wiring it into the UI. Generation calls a real LLM if `OPENAI_API_KEY` is set; otherwise it prints the constructed prompt.

In [1]:
import os
import pandas as pd
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

kb = pd.read_csv("support_kb_articles.csv")
model = SentenceTransformer("all-MiniLM-L6-v2")
doc_embeddings = model.encode(kb["content"].tolist())
print(f"Loaded {len(kb)} articles.")

/Users/suman/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/suman/Library/Python/3.9/lib/python/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loaded 20 articles.


## 1. Retrieval + prompt construction (same as last session)

In [2]:
def retrieve(query, top_k=3, min_similarity=0.25):
    query_embedding = model.encode([query])
    scores = cosine_similarity(query_embedding, doc_embeddings)[0]
    kb_scored = kb.copy()
    kb_scored["similarity"] = scores
    top = kb_scored.sort_values("similarity", ascending=False).head(top_k)
    return top[top["similarity"] >= min_similarity]

def build_prompt(query, context_docs):
    context_text = "\n\n".join(
        f"[{row.doc_id}] {row.title}: {row.content}" for row in context_docs.itertuples()
    )
    return f"""Answer the customer's question using ONLY the context below. If the context doesn't
fully answer the question, say so rather than guessing. Cite the doc_id(s) you used.

Context:
{context_text}

Question: {query}"""

## 2. Generation, with graceful offline fallback

In [3]:
def generate_answer(query, top_k=3, min_similarity=0.25):
    context_docs = retrieve(query, top_k=top_k, min_similarity=min_similarity)
    if context_docs.empty:
        return "No relevant knowledge-base article found for this question.", context_docs

    prompt = build_prompt(query, context_docs)

    if os.environ.get("OPENAI_API_KEY"):
        from openai import OpenAI
        client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])
        response = client.chat.completions.create(
            model="gpt-4", messages=[{"role": "user", "content": prompt}], temperature=0.2
        )
        return response.choices[0].message.content, context_docs
    else:
        return f"[No OPENAI_API_KEY — showing constructed prompt]\n\n{prompt}", context_docs

answer, sources = generate_answer("What is the return window for standard items?")
print(answer)
print("\nSources:")
print(sources[["doc_id", "title", "similarity"]])

[No OPENAI_API_KEY — showing constructed prompt]

Answer the customer's question using ONLY the context below. If the context doesn't
fully answer the question, say so rather than guessing. Cite the doc_id(s) you used.

Context:
[KB005] Return Window for Standard Items: Most items can be returned within 7 days of delivery, provided the original packaging and tags are intact. Perishable goods and personal care items are not eligible for return once opened.

[KB008] Exchange Instead of Refund: Customers may request a direct exchange instead of a refund for size or color mismatches, subject to stock availability at the same store. Exchanges follow the same 7-day return window as refunds.

[KB006] Refund Processing Time: Refunds for approved returns are processed within 5 to 7 business days after the returned item passes a quality check at the warehouse. Refunds are credited to the original payment method.

Question: What is the return window for standard items?

Sources:
  doc_id         

/Users/suman/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/suman/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/suman/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b


## 3. Test set — beyond the happy path

Before wiring this logic into the Streamlit app, run it against exact-match, paraphrase, multi-topic, and out-of-scope queries, exactly as the lecture script's evaluation table describes.

In [4]:
test_cases = [
    ("What is the return window for standard items?", "exact-wording match"),
    ("How long do I have to send something back?", "paraphrase, zero word overlap"),
    ("My order is late and I want to know if I can still get a refund", "spans two topics"),
    ("Do you sell electronics accessories", "out of scope entirely"),
]

for query, label in test_cases:
    print(f"\n=== [{label}] {query} ===")
    _, sources = generate_answer(query)
    if sources.empty:
        print("  -> No relevant article found (correct for out-of-scope)")
    else:
        for row in sources.itertuples():
            print(f"  -> {row.similarity:.3f} — {row.doc_id}: {row.title}")


=== [exact-wording match] What is the return window for standard items? ===
  -> 0.431 — KB005: Return Window for Standard Items
  -> 0.360 — KB008: Exchange Instead of Refund
  -> 0.354 — KB006: Refund Processing Time

=== [paraphrase, zero word overlap] How long do I have to send something back? ===
  -> 0.652 — KB005: Return Window for Standard Items
  -> 0.523 — KB007: Damaged Item on Arrival
  -> 0.487 — KB006: Refund Processing Time

=== [spans two topics] My order is late and I want to know if I can still get a refund ===
  -> 0.623 — KB008: Exchange Instead of Refund
  -> 0.547 — KB006: Refund Processing Time
  -> 0.472 — KB005: Return Window for Standard Items

=== [out of scope entirely] Do you sell electronics accessories ===
  -> No relevant article found (correct for out-of-scope)


/Users/suman/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/suman/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/suman/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b
/Users/suman/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/suman/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/suman/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b
/Users/suman/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: Run

## 4. Try it yourself

Once this logic behaves correctly across the test set above, it's ready to be wired into the Streamlit UI exactly as shown in `Building a RAG Application - app.py`. Try one more query of your own below.

In [5]:
my_query = "Can I exchange an item instead of getting a refund?"
answer, sources = generate_answer(my_query)
print(answer)
print("\nSources:")
print(sources[["doc_id", "title", "similarity"]] if not sources.empty else "none")

[No OPENAI_API_KEY — showing constructed prompt]

Answer the customer's question using ONLY the context below. If the context doesn't
fully answer the question, say so rather than guessing. Cite the doc_id(s) you used.

Context:
[KB008] Exchange Instead of Refund: Customers may request a direct exchange instead of a refund for size or color mismatches, subject to stock availability at the same store. Exchanges follow the same 7-day return window as refunds.

[KB006] Refund Processing Time: Refunds for approved returns are processed within 5 to 7 business days after the returned item passes a quality check at the warehouse. Refunds are credited to the original payment method.

[KB005] Return Window for Standard Items: Most items can be returned within 7 days of delivery, provided the original packaging and tags are intact. Perishable goods and personal care items are not eligible for return once opened.

Question: Can I exchange an item instead of getting a refund?

Sources:
  doc_id   

/Users/suman/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/suman/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/suman/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b
